# ChiliEdgeNet-Full — (1/3) Train & Baseline

Notebook **train** 2 model chinh: Scenario 1 (baseline) + Scenario 2
(domain-robust augmentation = model de xuat **ChiliEdgeNet**).

Scenario 2 mo phong anh cross-dataset co do phan giai thap, mo, thay doi ti le
doi tuong va anh sang. `BACKBONE_CUT=19` dung toan bo MobileNetV2 de kiem tra anh huong cua viec
cat backbone trong ban C14.

**Output quan trong:** `cfull_s1_seed123.pth`, `cfull_s2_seed123.pth` va
`chiliedgenet_full_s2_seed123_complete.pt`.
Sau khi chay xong -> **Save Version**, roi Add file `.pt` lam Input cho notebook
Cross-dataset Evaluation.

Cac phan phan tich (nhieu / XAI / latency / ablation / nen) tach sang NB2, NB3.

> **Ablation full backbone:** notebook nay lay toan bo 19 feature blocks cua MobileNetV2. Hay so sanh ket qua noi bo va cross-dataset voi ban C14; khong ghi de checkpoint cua C14.


In [ ]:
!pip install -q torchinfo

In [ ]:
import os, gc, warnings, math, random
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from PIL import Image, ImageFile
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchinfo import summary as torchinfo_summary
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# FGVC8 co mot so JPEG thieu vai byte o cuoi; Pillow van co the doc phan anh hop le.
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Chi can doi gia tri nay de chay mot seed moi.
SEED = 123

def set_seed(seed=SEED):
    """Co dinh Python / NumPy / PyTorch CPU + CUDA de tai lap."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE} | seed={SEED}')

In [ ]:
# Config
DATASET_PATH = '/kaggle/input/datasets/tminhhi/ot-21-06/dataset_smartcrop_plit_300'
TRAIN_DIR = os.path.join(DATASET_PATH, 'train')
VAL_DIR   = os.path.join(DATASET_PATH, 'val')
TEST_DIR  = os.path.join(DATASET_PATH, 'test')

IMG_SIZE, BATCH_SIZE = 224, 32
EPOCHS = 30
LR_BACKBONE, LR_NEW = 1e-4, 1e-3
PRETRAINED = True
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

LABEL_SMOOTHING = 0.05
WARMUP_FRAC     = 0.1

# Tiny-LLM Transformer: reduce 1280 -> 128, 1 khoi, 49 token
TF_EMBED_DIM, TF_DEPTH = 128, 1
TF_HEADS, TF_KV_HEADS = 4, 2
TF_MLP_RATIO, TF_DROP, TF_ROPE_BASE = 2.0, 0.1, 100.0

# Full backbone MobileNetV2:
# features[0] duoc tach thanh first_conv; features[1:19] lay TOAN BO phan con lai.
# Dau ra: 1280 kenh @ 7x7 = 49 token; khong cat bo block nao cua MobileNetV2.
BACKBONE_CUT = 19

# Muc nhieu ap o test-time (khong co mat luc train)
EVAL_SETS_CFG = ['clean', 'blur2', 'dark']   # them 'noise'/'jpeg' neu muon
BLUR_K, DARK_SCALE = 7, 0.4
ISO_SHIFT, ISO_INT = 0.02, 0.1
JPEG_Q = 30

In [ ]:
# ===== Few-shot join tu bo ngoai (ChiliBD) — CHI join vao TRAIN, KHONG dung vao VAL/TEST =====
# Dataset Kaggle 'chili-shot-{N}' KHONG co train/val/test, chi co class folders truc tiep
# trong subfolder '{N}shot' (vd chili-shot-5/5shot/anthracnose/...) -> chi dung lam nguon join train.
USE_FEWSHOT_JOIN = True
FEWSHOT_SHOT = 5   # so anh moi class lay tu bo ngoai: 5 / 10 / 20 (khop ten dataset da upload)
FEWSHOT_DATASET_PATH = f'/kaggle/input/chili-shot-{FEWSHOT_SHOT}'
FEWSHOT_DIR = os.path.join(FEWSHOT_DATASET_PATH, f'{FEWSHOT_SHOT}shot')

if USE_FEWSHOT_JOIN:
    if not os.path.isdir(FEWSHOT_DIR):
        raise FileNotFoundError(
            f'Khong thay FEWSHOT_DIR={FEWSHOT_DIR}. Kiem tra dataset chili-shot-{FEWSHOT_SHOT} '
            f'da duoc add vao Input chua, hoac dat USE_FEWSHOT_JOIN=False.')
    import shutil
    MERGED_TRAIN_DIR = '/kaggle/working/train_fewshot_join'
    if os.path.exists(MERGED_TRAIN_DIR):
        shutil.rmtree(MERGED_TRAIN_DIR)
    shutil.copytree(TRAIN_DIR, MERGED_TRAIN_DIR)    # copy toan bo train goc cua bo A
    _n_added = 0
    for _cls in os.listdir(FEWSHOT_DIR):
        _src_dir = os.path.join(FEWSHOT_DIR, _cls)
        _dst_dir = os.path.join(MERGED_TRAIN_DIR, _cls)
        os.makedirs(_dst_dir, exist_ok=True)        # phong khi class chua ton tai o bo A
        for _fname in os.listdir(_src_dir):
            shutil.copy2(os.path.join(_src_dir, _fname), os.path.join(_dst_dir, _fname))
            _n_added += 1
    TRAIN_DIR = MERGED_TRAIN_DIR   # CHI doi TRAIN_DIR; VAL_DIR/TEST_DIR giu nguyen bo A goc, khong bi dung
    print(f'[Few-shot] da join {_n_added} anh ({FEWSHOT_SHOT}-shot/class) tu {FEWSHOT_DIR} vao TRAIN_DIR')
    print(f'[Few-shot] TRAIN_DIR moi: {TRAIN_DIR}')
    print(f'[Few-shot] VAL_DIR/TEST_DIR khong doi: {VAL_DIR} | {TEST_DIR}')
else:
    print('[Few-shot] USE_FEWSHOT_JOIN=False -> dung TRAIN_DIR goc cua bo A, khong join gi.')

In [ ]:
# Dataset: class names + distribution + class weights
_ds = ImageFolder(TRAIN_DIR)
class_names = _ds.classes
NUM_CLASSES = len(class_names)
counts = np.bincount([l for _, l in _ds.samples], minlength=NUM_CLASSES)

import pandas as pd
_dist = {}
for _split, _d in [('train', TRAIN_DIR), ('val', VAL_DIR), ('test', TEST_DIR)]:
    _n = np.bincount([l for _, l in ImageFolder(_d).samples], minlength=NUM_CLASSES)
    _dist[_split] = _n
dist_df = pd.DataFrame(_dist, index=class_names)
dist_df['total'] = dist_df.sum(axis=1)
dist_df.loc['TONG'] = dist_df.sum()
print(dist_df.to_string())
dist_df.to_csv('cfull_class_distribution.csv')

fig, ax = plt.subplots(figsize=(max(8, NUM_CLASSES * 0.9), 4))
dist_df.drop(index='TONG')[['train', 'val', 'test']].plot(kind='bar', stacked=True, ax=ax)
ax.set_ylabel('So anh'); plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.savefig('cfull_class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'NUM_CLASSES={NUM_CLASSES} | {class_names}')
ratio = counts.max() / counts.min()
if ratio >= 1.5:
    cw = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES),
                              y=np.array([l for _, l in _ds.samples]))
    CLASS_WEIGHTS = torch.tensor(cw, dtype=torch.float32)
    print(f'Imbalanced (ratio={ratio:.2f}) -> class weights ON')
else:
    CLASS_WEIGHTS = None
    print(f'Balanced (ratio={ratio:.2f}) -> no weighting')
del _ds

In [ ]:
# ===== TINY-LLM TRANSFORMER (RMSNorm + RoPE-2D + GQA) — dong gop chinh cua model =====
class RMSNorm(nn.Module):
    """Chuan hoa theo RMS (khong tru mean nhu LayerNorm) — nhe hon, dung trong LLM (LLaMA)."""
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.eps, self.weight = eps, nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def build_axial_rope(gh, gw, head_dim, base=100.0):
    """RoPE-2D: sinh cos/sin ma hoa vi tri (hang, cot) cua luoi token bang phep xoay.
    Nua dau head_dim cho toa do y (hang), nua sau cho x (cot). 0 tham so hoc.
    Tra (cos_y, sin_y, cos_x, sin_x)."""
    d = head_dim // 2
    assert d % 2 == 0, 'head_dim phai chia het 4'
    inv = 1.0 / (base ** (torch.arange(0, d, 2).float() / d))
    gy = torch.arange(gh).float().view(gh, 1).expand(gh, gw).reshape(-1)
    gx = torch.arange(gw).float().view(1, gw).expand(gh, gw).reshape(-1)
    ey = torch.cat([torch.outer(gy, inv)] * 2, dim=-1)
    ex = torch.cat([torch.outer(gx, inv)] * 2, dim=-1)
    return ey.cos()[None,None], ey.sin()[None,None], ex.cos()[None,None], ex.sin()[None,None]

def _rot_half(x):
    """Phep xoay cua RoPE trong khong gian embedding: [x1, x2] -> [-x2, x1]."""
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rope2d(t, cy, sy, cx, sx):
    """Ap RoPE-2D: nua dau embedding xoay theo y, nua sau xoay theo x."""
    d = t.shape[-1] // 2
    ty, tx = t[..., :d], t[..., d:]
    ty = ty * cy + _rot_half(ty) * sy
    tx = tx * cx + _rot_half(tx) * sx
    return torch.cat([ty, tx], dim=-1)

class GroupedQueryAttention(nn.Module):
    """Attention kieu LLM: nh query head dung chung nkv KV head (nh > nkv)
    -> giam tham so & bo nho so voi Multi-Head Attention chuan. O day nh=4, nkv=2."""
    def __init__(self, dim, nh, nkv, dropout=0.0):
        super().__init__()
        assert dim % nh == 0 and nh % nkv == 0
        self.nh, self.nkv, self.hd = nh, nkv, dim // nh
        self.groups, self.scale = nh // nkv, (dim // nh) ** -0.5
        self.q = nn.Linear(dim, nh * self.hd, bias=False)
        self.k = nn.Linear(dim, nkv * self.hd, bias=False)
        self.v = nn.Linear(dim, nkv * self.hd, bias=False)
        self.o = nn.Linear(nh * self.hd, dim, bias=False)
        self.ad, self.pd = nn.Dropout(dropout), nn.Dropout(dropout)
    def forward(self, x, rope):
        B, N, _ = x.shape; cy, sy, cx, sx = rope
        q = self.q(x).view(B, N, self.nh, self.hd).transpose(1, 2)
        k = self.k(x).view(B, N, self.nkv, self.hd).transpose(1, 2)
        v = self.v(x).view(B, N, self.nkv, self.hd).transpose(1, 2)
        q, k = apply_rope2d(q, cy, sy, cx, sx), apply_rope2d(k, cy, sy, cx, sx)  # gan vi tri vao q,k
        if self.groups > 1:                              # lap KV cho du so query head
            k = k.repeat_interleave(self.groups, dim=1)
            v = v.repeat_interleave(self.groups, dim=1)
        attn = self.ad(((q @ k.transpose(-2, -1)) * self.scale).softmax(-1))
        return self.pd(self.o((attn @ v).transpose(1, 2).reshape(B, N, -1)))

class TinyLLMBlock(nn.Module):
    """1 khoi transformer kieu LLM: pre-RMSNorm + GQA(RoPE) residual, roi pre-RMSNorm + MLP(GELU) residual."""
    def __init__(self, dim, nh, nkv, mlp_ratio=2.0, dropout=0.1):
        super().__init__()
        self.n1 = RMSNorm(dim); self.attn = GroupedQueryAttention(dim, nh, nkv, dropout)
        self.n2 = RMSNorm(dim); h = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, h), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(h, dim), nn.Dropout(dropout))
    def forward(self, x, rope):
        x = x + self.attn(self.n1(x), rope)
        return x + self.mlp(self.n2(x))

# ===== MODEL DE XUAT: ChiliEdgeNet =====
class ChiliEdgeNet(nn.Module):
    """Toan bo MobileNetV2 features[0:19] trich dac trung -> Conv1x1 giam chieu ve embed_dim
    -> chuoi 196 token (14x14) -> depth khoi Tiny-LLM Transformer -> pool trung binh -> phan lop."""
    def __init__(self, num_classes, pretrained=True, cut=BACKBONE_CUT,
                 embed_dim=TF_EMBED_DIM, depth=TF_DEPTH, heads=TF_HEADS, kv_heads=TF_KV_HEADS,
                 mlp_ratio=TF_MLP_RATIO, drop=TF_DROP, rope_base=TF_ROPE_BASE):
        super().__init__()
        w = torchvision.models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None
        base = torchvision.models.mobilenet_v2(weights=w)
        self.first_conv = base.features[0]
        self.backbone = nn.Sequential(*list(base.features[1:cut]))
        # suy ra so kenh + kich thuoc luoi ra cua backbone (chay thu 1 tensor rong)
        with torch.no_grad():
            feat = self.backbone(self.first_conv(torch.zeros(1,3,IMG_SIZE,IMG_SIZE)))
        c_out, grid = feat.shape[1], feat.shape[2]
        self.reduce = nn.Sequential(nn.Conv2d(c_out, embed_dim, 1, bias=False),   # giam kenh -> embed_dim
                                    nn.BatchNorm2d(embed_dim), nn.GELU())
        self.emb_drop = nn.Dropout(drop)
        cy, sy, cx, sx = build_axial_rope(grid, grid, embed_dim // heads, base=rope_base)
        for n, b in zip(['rcy','rsy','rcx','rsx'], [cy, sy, cx, sx]):
            self.register_buffer(n, b, persistent=False)     # RoPE la buffer, khong phai tham so hoc
        self.encoder = nn.ModuleList([TinyLLMBlock(embed_dim, heads, kv_heads, mlp_ratio, drop)
                                      for _ in range(depth)])
        self.norm = RMSNorm(embed_dim)
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(embed_dim, num_classes)
        print(f'[ChiliEdgeNet] backbone out: {c_out}ch @ {grid}x{grid} -> {grid*grid} token')
    def forward(self, x):
        x = self.backbone(self.first_conv(x))                            # CNN trich dac trung
        x = self.emb_drop(self.reduce(x).flatten(2).transpose(1, 2))     # (B,C,H,W) -> (B, HxW token, embed)
        rope = (self.rcy, self.rsy, self.rcx, self.rsx)
        for blk in self.encoder:
            x = blk(x, rope)
        return self.fc(self.drop(self.norm(x).mean(dim=1)))              # pool token -> phan lop

print('ChiliEdgeNet loaded.')

def build_main():
    """Tao model full-backbone: cut=19 (toan bo MobileNetV2) + Tiny-LLM Transformer."""
    return ChiliEdgeNet(NUM_CLASSES, pretrained=PRETRAINED).to(DEVICE)

In [ ]:
_m = build_main()
torchinfo_summary(_m, input_size=(1, 3, IMG_SIZE, IMG_SIZE),
                  col_names=['input_size', 'output_size', 'num_params'],
                  depth=4, device=DEVICE, verbose=1)
del _m

In [ ]:
# ===== Helpers: seed, loss/optimizer/scheduler, vong train & eval, bao cao =====
import math, random

SEED = 42

def set_seed(seed=SEED):
    """Co dinh moi nguon ngau nhien (Python / NumPy / PyTorch CPU + CUDA) de tai lap."""
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def make_criterion():
    """CrossEntropy + label smoothing; tu gan class_weights neu dataset lech lop."""
    weight = CLASS_WEIGHTS.to(DEVICE) if CLASS_WEIGHTS is not None else None
    return nn.CrossEntropyLoss(weight=weight, label_smoothing=LABEL_SMOOTHING)

def make_optimizer(model):
    """Adam 2 nhom learning-rate: backbone (pretrained) hoc cham, phan moi hoc nhanh."""
    backbone_prefix = ('first_conv', 'backbone')
    backbone_params = [p for name, p in model.named_parameters()
                       if p.requires_grad and name.startswith(backbone_prefix)]
    new_params      = [p for name, p in model.named_parameters()
                       if p.requires_grad and not name.startswith(backbone_prefix)]
    return torch.optim.Adam(
        [{'params': backbone_params, 'lr': LR_BACKBONE},
         {'params': new_params,      'lr': LR_NEW}], lr=LR_NEW)

def make_scheduler(optimizer, epochs):
    """Linear warmup (WARMUP_FRAC epoch dau) roi cosine decay ve 0."""
    warmup = max(1, int(WARMUP_FRAC * epochs))
    def lr_scale(epoch):                              # epoch: 0-indexed
        if epoch < warmup:
            return (epoch + 1) / warmup               # tang tuyen tinh
        progress = (epoch - warmup) / max(1, epochs - warmup)
        return 0.5 * (1 + math.cos(math.pi * progress))   # cosine giam dan
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_scale)

def train_one_epoch(model, loader, optimizer, criterion):
    """Chay 1 epoch train; tra ve (loss trung binh, accuracy) tren tap train."""
    model.train()
    total_loss = correct = seen = 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        seen       += images.size(0)
    return total_loss / seen, correct / seen

@torch.no_grad()
def evaluate(model, loader, criterion):
    """Danh gia khong cap nhat gradient; tra ve (loss, accuracy, du_doan, nhan_that)."""
    model.eval()
    total_loss = correct = seen = 0
    preds, labels_all = [], []
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits = model(images)
        total_loss += criterion(logits, labels).item() * images.size(0)
        pred = logits.argmax(1)
        correct += (pred == labels).sum().item()
        seen    += images.size(0)
        preds      += pred.cpu().tolist()
        labels_all += labels.cpu().tolist()
    return total_loss / seen, correct / seen, preds, labels_all

def train_model(model, train_loader, val_loader, criterion, ckpt_path, epochs=None):
    """Train + validate moi epoch, luu checkpoint TOT NHAT theo val_acc, reload truoc khi tra ve."""
    epochs = epochs or EPOCHS
    optimizer = make_optimizer(model)
    scheduler = make_scheduler(optimizer, epochs)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = 0.0
    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()
        for key, value in zip(history, [train_loss, val_loss, train_acc, val_acc]):
            history[key].append(value)
        print(f'Epoch {epoch:2d}/{epochs} | train {train_acc:.4f} | val {val_acc:.4f}')
        if val_acc > best_val_acc:                    # chi luu khi val_acc cai thien
            best_val_acc = val_acc
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))   # reload best truoc khi test
    return history

def count_params(model):
    """Tong so tham so cua model."""
    return sum(p.numel() for p in model.parameters())

def report(tag, model, loader):
    """In accuracy / F1-macro / recall-macro / params + classification_report per-class."""
    _, acc, preds, labels_all = evaluate(model, loader, make_criterion())
    f1     = f1_score(labels_all, preds, average='macro')
    recall = recall_score(labels_all, preds, average='macro')
    print(f'[{tag}] acc={acc:.4f} f1={f1:.4f} recall={recall:.4f} params={count_params(model):,}')
    print(classification_report(labels_all, preds, target_names=class_names, digits=4))
    return acc, f1, preds, labels_all

def plot_cm(labels_all, preds, title, path, cmap='Blues'):
    """Ve + luu confusion matrix (dpi=300)."""
    cm = confusion_matrix(labels_all, preds)
    fig, ax = plt.subplots(figsize=(max(6, NUM_CLASSES), max(5, NUM_CLASSES - 1)))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.tight_layout(); plt.savefig(path, dpi=300, bbox_inches='tight'); plt.show()

def plot_curve(history, title, path):
    """Ve + luu learning curve (loss & accuracy theo epoch)."""
    epochs_range = range(1, len(history['train_loss']) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    ax[0].plot(epochs_range, history['train_loss'], label='train')
    ax[0].plot(epochs_range, history['val_loss'], label='val')
    ax[0].set_title('Loss'); ax[0].legend(); ax[0].grid(alpha=0.3)
    ax[1].plot(epochs_range, history['train_acc'], label='train')
    ax[1].plot(epochs_range, history['val_acc'], label='val')
    ax[1].set_title('Accuracy'); ax[1].legend(); ax[1].grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(path, dpi=300, bbox_inches='tight'); plt.show()

print('Helpers loaded (label smoothing + cosine warmup).')

## Scenario 1 — Baseline (Resize + Normalize)

In [ ]:
# Seed cua lan chay hien tai; doi gia tri nay khi lap thuc nghiem.
SEED = 123

tf_plain = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])
train_loader1 = DataLoader(ImageFolder(TRAIN_DIR, tf_plain), BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader1   = DataLoader(ImageFolder(VAL_DIR,   tf_plain), BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader1  = DataLoader(ImageFolder(TEST_DIR,  tf_plain), BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

set_seed(SEED)
model_s1 = build_main()
set_seed(SEED)   # seed lai truoc khi train: khop dung quy trinh cua cac bien the ablation
ckpt_s1 = f'cfull_s1_seed{SEED}.pth'
hist_s1 = train_model(model_s1, train_loader1, val_loader1, make_criterion(), ckpt_s1)
s1_acc, s1_f1, s1_preds, s1_labs = report(f'Full-S1 seed={SEED}', model_s1, test_loader1)

In [ ]:
plot_cm(s1_labs, s1_preds, 'Confusion Matrix - Full S1', 'cfull_cm_s1.png', 'Blues')

In [ ]:
plot_curve(hist_s1, 'Full - Scenario 1', 'cfull_curve_s1.png')

In [ ]:
torch.cuda.empty_cache(); gc.collect()

## Scenario 2 — Domain-robust augmentation

In [ ]:
class RandomLowResolution:
    """Mo phong thumbnail/chat luong thap: downsample roi upscale ve IMG_SIZE.

    Phep bien doi nay khong tao them chi tiet; no buoc model hoc dac trung ben
    vung hon khi anh ngoai chi co 64-160 pixel.
    """
    def __init__(self, min_size=64, max_size=160):
        self.min_size = min_size
        self.max_size = max_size

    def __call__(self, image):
        size = random.randint(self.min_size, self.max_size)
        image = image.resize((size, size), Image.Resampling.BILINEAR)
        return image.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)


tf_aug = T.Compose([
    # Thay Resize co dinh: thay doi ti le doi tuong trong khung hinh.
    T.RandomResizedCrop(
        IMG_SIZE,
        scale=(0.55, 1.0),
        ratio=(0.75, 1.33),
        interpolation=T.InterpolationMode.BILINEAR,
    ),

    # Mo phong anh web/thumbnail cua cross-dataset (chi ap ngau nhien 40%).
    T.RandomApply([RandomLowResolution(64, 160)], p=0.40),
    T.RandomApply([T.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))], p=0.25),

    # Bien doi hinh hoc va mau vua phai, tranh lam bien dang trieu chung qua muc.
    T.RandomRotation(20),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),

    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

train_loader2 = DataLoader(
    ImageFolder(TRAIN_DIR, tf_aug),
    BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
)
val_loader2, test_loader2 = val_loader1, test_loader1

set_seed(SEED)
model_s2 = build_main()
set_seed(SEED)   # seed lai ngay truoc train de tai lap dung thi nghiem
ckpt_s2 = f'cfull_s2_seed{SEED}.pth'
hist_s2 = train_model(
    model_s2, train_loader2, val_loader2, make_criterion(), ckpt_s2
)
s2_acc, s2_f1, s2_preds, s2_labs = report(
    f'Full-S2 robust seed={SEED}', model_s2, test_loader2
)

# train_model() da reload checkpoint co val_acc tot nhat truoc khi export.
example_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
with torch.no_grad():
    scripted_s2 = torch.jit.trace(model_s2.eval(), example_input)
scripted_s2.save(f'chiliedgenet_full_s2_seed{SEED}_complete.pt')
print(f'Da luu chiliedgenet_full_s2_seed{SEED}_complete.pt')

In [ ]:
# ===== Verify exported .pt on the original test set =====
# Muc dich: xac nhan TorchScript .pt cho ket qua giong model vua train/reload .pth.
PT_PATH = f'chiliedgenet_full_s2_seed{SEED}_complete.pt'

pt_model = torch.jit.load(PT_PATH, map_location=DEVICE)
pt_model.eval()

pt_loss, pt_acc, pt_preds, pt_labs = evaluate(
    pt_model, test_loader2, make_criterion()
)
pt_f1 = f1_score(pt_labs, pt_preds, average='macro')
pt_recall = recall_score(pt_labs, pt_preds, average='macro')

prediction_agreement = np.mean(
    np.asarray(pt_preds) == np.asarray(s2_preds)
)

# So sanh logits tren mot batch de phat hien loi export/load am tham.
verify_images, _ = next(iter(test_loader2))
verify_images = verify_images.to(DEVICE)
with torch.inference_mode():
    logits_live = model_s2.eval()(verify_images)
    logits_pt = pt_model(verify_images)
max_logit_diff = (logits_live - logits_pt).abs().max().item()

print(f'[PT VERIFY] path={PT_PATH}')
print(f'[PT VERIFY] loss={pt_loss:.4f} | acc={pt_acc:.4f} | '
      f'f1={pt_f1:.4f} | recall={pt_recall:.4f}')
print(f'[PT VERIFY] prediction agreement with live/.pth model: '
      f'{prediction_agreement * 100:.2f}%')
print(f'[PT VERIFY] max absolute logit difference: {max_logit_diff:.8f}')

if prediction_agreement == 1.0 and abs(pt_acc - s2_acc) < 1e-12:
    print('[PT VERIFY] PASS: .pt va model best-checkpoint cho cung ket qua test.')
else:
    print('[PT VERIFY] WARNING: ket qua .pt khac model trong bo nho — can kiem tra export.')

del verify_images, logits_live, logits_pt
torch.cuda.empty_cache()


In [ ]:
plot_cm(s2_labs, s2_preds, 'Confusion Matrix - Full S2', 'cfull_cm_s2.png', 'Greens')

In [ ]:
plot_curve(hist_s2, 'Full - Scenario 2', 'cfull_curve_s2.png')

In [ ]:
import pandas as pd
df = pd.DataFrame([
    {'scenario': 'S1 baseline',  'params': count_params(model_s1), 'acc': s1_acc, 'f1': s1_f1},
    {'scenario': 'S2 domain-robust', 'params': count_params(model_s2), 'acc': s2_acc, 'f1': s2_f1},
])
print(df.to_string(index=False)); df.to_csv('cfull_results.csv', index=False)
torch.cuda.empty_cache(); gc.collect()